In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from dotenv import load_dotenv

load_dotenv()

BASE_DIR = os.getenv("BASE_DIR")

directory = f"{BASE_DIR}/data/error_analysis/o"
MODEL_COLOR = {'cnn': '#eb6834', 'residual': '#1baf7a'}
MODEL_LABEL = {'cnn': 'CNN', 'residual': 'Residual'}
INK = '#52514e'
BINS = ['0', '1', '2', '3-5', '6-10', '11+']        # forgetting count

bins = pd.read_csv(f"{directory}/bins.csv")
gain = pd.read_csv(f"{directory}/gain_curve.csv")
summary = pd.read_csv(f"{directory}/summary.csv")
print(summary.groupby('model')[['mse_E', 'train_mse', 'spearman', 'top5_error_share', 'oracle_top5_error_share']].mean().round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1) gain curve: states sorted by forgetting count (high first) vs by true error (oracle)
ax = axes[0]
ax.plot([0, 1], [0, 1], color=INK, lw=0.8, ls=':')
for model, color in MODEL_COLOR.items():
    for sorted_by, ls in (('forgetting', '-'), ('oracle', '--')):
        g = gain[(gain.model == model) & (gain.sorted_by == sorted_by)]
        mean = g.groupby('state_share')['error_share'].mean()
        ax.plot(mean.index * 100, mean.values * 100, color=color, ls=ls, lw=1.8)
ax.set_xlim(0, 20); ax.set_ylim(0, 100)       # the top 20% of states is where it happens
ax.set_xlabel('States, sorted (% of E)'); ax.set_ylabel('Cumulative share of the error (%)')
ax.set_title('Where the error sits')
handles = ([Line2D([], [], color=c, lw=2, label=MODEL_LABEL[m]) for m, c in MODEL_COLOR.items()] +
           [Line2D([], [], color=INK, lw=2, ls='-', label='sorted by forgetting count'),
            Line2D([], [], color=INK, lw=2, ls='--', label='sorted by true error (oracle)'),
            Line2D([], [], color=INK, lw=1, ls=':', label='uniform')])
ax.legend(handles=handles, frameon=False, fontsize=9, loc='lower right')

# 2) per forgetting-count bin: share of the states vs share of the error
ax = axes[1]
a = bins[bins.subset == 'all']
x = np.arange(len(BINS)); w = 0.27
state_share = a[a.model == 'cnn'].groupby('bin')['state_share'].mean().reindex(BINS) * 100
ax.bar(x - w, state_share.values, w, color='#b9b8b4', label='share of the states')
for i, (model, color) in enumerate(MODEL_COLOR.items()):
    g = a[a.model == model].groupby('bin')['error_share']
    ax.bar(x + i * w, g.mean().reindex(BINS) * 100, w, yerr=g.std().reindex(BINS) * 100, color=color,
           error_kw=dict(lw=0.8, capsize=2), label=f'share of the error, {MODEL_LABEL[model]}')
ax.set_xticks(x); ax.set_xticklabels(BINS)
ax.set_xlabel('Forgetting count'); ax.set_ylabel('% of E / % of the total error')
ax.set_title('State share vs error share')
ax.legend(frameon=False, fontsize=9)

# 3) mean error per bin, states inside the training subset S vs outside it
ax = axes[2]
for model, color in MODEL_COLOR.items():
    for subset, ls, marker in (('in_S', '-', 'o'), ('not_in_S', '--', 's')):
        g = bins[(bins.model == model) & (bins.subset == subset)].groupby('bin')['mean_error']
        ax.errorbar(x, g.mean().reindex(BINS), yerr=g.std().reindex(BINS), color=color, ls=ls, marker=marker,
                    ms=4, lw=1.6, capsize=2, elinewidth=0.8)
ax.axhline(1.0, color=INK, lw=0.8, ls=':')        # predicting 0 on a +-1 label
ax.text(len(BINS) - 1, 1.02, 'predicting 0', ha='right', fontsize=8, color=INK)
ax.set_xticks(x); ax.set_xticklabels(BINS)
ax.set_xlabel('Forgetting count'); ax.set_ylabel('Mean squared error')
ax.set_title('Error inside S (solid) vs outside S (dashed)')
handles = [Line2D([], [], color=c, lw=2, label=MODEL_LABEL[m]) for m, c in MODEL_COLOR.items()] + \
          [Line2D([], [], color=INK, lw=2, ls='-', marker='o', ms=4, label='in S (trained on)'),
           Line2D([], [], color=INK, lw=2, ls='--', marker='s', ms=4, label='not in S')]
ax.legend(handles=handles, frameon=False, fontsize=9, loc='upper left')

for ax in axes:
    ax.grid(alpha=0.3, lw=0.5)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
fig.suptitle('Error on E by forgetting count, herding 10% of o.txt (mean over seeds, bars = std)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

directory = "/home/axelbjarkar/loka_local/data/error_analysis/o"   # or .../T-404-LOKA-code/data/error_analysis/o for the cluster files
MODEL = 'cnn'                       # model shown in the gain-curve panel
MODEL_LABEL = {'cnn': 'CNN', 'residual': 'Residual'}
INK = '#52514e'
SORTER_STYLE = {                    # sorter -> (label, colour, linestyle)
    'cal_raw':             ('CAL, raw positions',     '#2a78d6', '-'),
    'cal_penultimate':     ('CAL, penultimate layer', '#eb6834', '-'),
    'forgetting (oracle)': ('forgetting count (oracle)', '#1baf7a', '--'),
    'true error (oracle)': ('true error (oracle)',    INK,       '--'),
    'random':              ('random',                 '#b9b8b4', ':'),
}

summary = pd.read_csv(f"{directory}/cal_summary.csv")
gain = pd.read_csv(f"{directory}/cal_gain_curve.csv")
sorters = [s for s in SORTER_STYLE if s in set(summary.sorter)]
models = [m for m in MODEL_LABEL if m in set(summary.model)]
print(summary.groupby(['model', 'sorter'], sort=False)[['spearman', 'top1_error_share', 'top5_error_share',
                                                          'top5_precision_hard', 'top5_recall_hard']].mean().round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1) gain curve on the pool (E minus S): how much of the error do the top-x% of states hold?
ax = axes[0]
for s in sorters:
    label, color, ls = SORTER_STYLE[s]
    g = gain[(gain.model == MODEL) & (gain.sorted_by == s)].groupby('state_share')['error_share'].mean()
    ax.plot(g.index * 100, g.values * 100, color=color, ls=ls, lw=1.8, label=label)
ax.set_xlim(0, 20); ax.set_ylim(0, 100)
ax.set_xlabel('Top x% of the pool (E minus S)'); ax.set_ylabel('Share of the pool error (%)')
ax.set_title(f'Where the error is, by score ({MODEL_LABEL[MODEL]})')
ax.legend(frameon=False, fontsize=9, loc='lower right')

# 2) share of the pool error held by the top 5% (random = 5%)
ax = axes[1]
x = np.arange(len(sorters)); w = 0.8 / len(models)
for i, m in enumerate(models):
    g = summary[summary.model == m].groupby('sorter')['top5_error_share']
    mean, std = g.mean().reindex(sorters) * 100, g.std().reindex(sorters).fillna(0) * 100
    ax.bar(x + (i - (len(models) - 1) / 2) * w, mean.values, w, yerr=std.values, error_kw=dict(lw=0.8, capsize=2),
           color=[SORTER_STYLE[s][1] for s in sorters], alpha=1 - 0.35 * i, label=MODEL_LABEL[m])
ax.axhline(5, color=INK, lw=0.8, ls=':'); ax.text(len(sorters) - 0.5, 5.8, 'random = 5%', ha='right', fontsize=8, color=INK)
ax.set_xticks(x); ax.set_xticklabels([SORTER_STYLE[s][0].replace(', ', ',\n').replace(' (oracle)', '\n(oracle)') for s in sorters], fontsize=8)
ax.set_ylabel('Error share of the top 5% (%)'); ax.set_title('Top 5% of the pool by each score')
if len(models) > 1:
    ax.legend(handles=[Line2D([], [], color=INK, lw=6, alpha=1 - 0.35 * i, label=MODEL_LABEL[m]) for i, m in enumerate(models)],
              frameon=False, fontsize=9)

# 3) are the picked states the hard ones? share of the top 5% with forgetting count >= 11 (base rate in the pool in grey)
ax = axes[2]
for i, m in enumerate(models):
    g = summary[summary.model == m].groupby('sorter')['top5_precision_hard']
    mean, std = g.mean().reindex(sorters) * 100, g.std().reindex(sorters).fillna(0) * 100
    ax.bar(x + (i - (len(models) - 1) / 2) * w, mean.values, w, yerr=std.values, error_kw=dict(lw=0.8, capsize=2),
           color=[SORTER_STYLE[s][1] for s in sorters], alpha=1 - 0.35 * i)
ax.axhline(5.4, color=INK, lw=0.8, ls=':'); ax.text(len(sorters) - 0.5, 6.2, 'base rate in E, 5.4%', ha='right', fontsize=8, color=INK)
ax.set_xticks(x); ax.set_xticklabels([SORTER_STYLE[s][0].replace(', ', ',\n').replace(' (oracle)', '\n(oracle)') for s in sorters], fontsize=8)
ax.set_ylabel('% of the top 5% with forgetting count >= 11'); ax.set_title('Do the scores pick the hard states?')

for a in axes:
    a.grid(alpha=0.3, lw=0.5)
    for side in ('top', 'right'):
        a.spines[side].set_visible(False)
fig.suptitle('Label-free CAL scores vs oracles on the unseen states, herding 10% of o.txt', fontsize=13)
plt.tight_layout()
plt.show()